In [1]:
import sys
import os
sys.path.append(os.path.realpath('../../'))

In [2]:
from tqdm.auto import tqdm
from typing import List
import re
from typing import List, Tuple, Dict
from data.dataset import GraphDataset, ReimburseGraphDataset, DataAugmentationLevel, DialogNode, NodeType
import nltk
from statistics import mean 

In [3]:
human_data_train = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
human_data_test = ReimburseGraphDataset('en/reimburse/test_graph.json', 'en/reimburse/test_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
generated_data_train_v1 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v1.json", resource_dir="../../resources/")
generated_data_train_v2 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v2.json", resource_dir="../../resources/")
generated_data_train_v3 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v3.json", resource_dir="../../resources/")

LOADING GRAPH...
GRAPH LOADED
LOADING ANSWERS FROM ../../resources/en/reimburse/train_answers.json...
- not using synonyms
ANSWERS LOADED
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  1
LOADING GRAPH...
GRAPH LOADED
LOADING ANSWERS FROM ../../resources/en/reimburse/test_answers.json...
- not using synonyms
ANSWERS LOADED
===== Dataset Statistics =====
- files:  en/reimburse/test_graph.json en/reimburse/test_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 173
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
LOADING GRAPH...
- Loading questions from  ../../resources/en/reimburse/generated/train

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
chencherry = SmoothingFunction()

def calculate_self_bleu(node: DialogNode, n_grams: int = 3) -> float:
    questions = set([q.text for q in node.questions])
    scores = []
    for hypothesis in questions:
        # take each generated sentence as hypothesis once and test against all other questions as references.
        references = questions.difference(set([hypothesis])) 
        scores.append(sentence_bleu(list(references), hypothesis, smoothing_function=chencherry.method1, weights=[1/n_grams for _ in range(n_grams)]))
    # take average as self-bleu
    return mean(scores)


In [5]:
datasets = {
    "Human Train": human_data_train,
    # "Human Test": human_data_test,
    "Gen V1": generated_data_train_v1,
    "Gen V2": generated_data_train_v2,
    "Gen V3": generated_data_train_v3
}

In [13]:
NGRAMS = [1,2,3,4,5]


ngram_scores = {}
full_scores = {}
for ngram in tqdm(NGRAMS):
    dataset_scores = {}
    full_scores[ngram] = {}
    for dataset_name in datasets:
        scores = []
        for node in datasets[dataset_name].nodes_by_type[NodeType.INFO]:
            if len(node.questions) <= 1:
                continue
            scores.append(calculate_self_bleu(node, ngram))
        full_scores[ngram][dataset_name] = scores
        dataset_scores[dataset_name] = mean(scores)
    ngram_scores[ngram] = dataset_scores

  0%|          | 0/5 [00:00<?, ?it/s]

In [8]:
import pprint

In [10]:
print("BLEU")
pprint.pprint(ngram_scores)

BLEU
{1: {'Gen V1': 0.9533895014749563,
     'Gen V2': 0.9490782132882306,
     'Gen V3': 0.8517171771986467,
     'Human Train': 0.7770562488537773},
 2: {'Gen V1': 0.9185073822750446,
     'Gen V2': 0.9049550760084472,
     'Gen V3': 0.776305989018358,
     'Human Train': 0.6821749528525342},
 3: {'Gen V1': 0.873387712154815,
     'Gen V2': 0.8505019196330329,
     'Gen V3': 0.7096502411590758,
     'Human Train': 0.600774251437181},
 4: {'Gen V1': 0.8340059382616397,
     'Gen V2': 0.8035380755031903,
     'Gen V3': 0.6597925198721522,
     'Human Train': 0.5397821639946383},
 5: {'Gen V1': 0.80045991481279,
     'Gen V2': 0.7634912301344376,
     'Gen V3': 0.6208092987870925,
     'Human Train': 0.49108430118472124}}


In [16]:
list(full_scores[1].keys())
from scipy.stats import ttest_ind, f_oneway, tukey_hsd

In [17]:
print("N-1")
print(tukey_hsd(full_scores[1]["Human Train"], full_scores[1]["Gen V1"],full_scores[1]["Gen V2"], full_scores[1]["Gen V3"]))

Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.176     0.000    -0.206    -0.147
 (0 - 2)     -0.172     0.000    -0.202    -0.142
 (0 - 3)     -0.075     0.000    -0.104    -0.045
 (1 - 0)      0.176     0.000     0.147     0.206
 (1 - 2)      0.004     0.981    -0.025     0.033
 (1 - 3)      0.102     0.000     0.072     0.131
 (2 - 0)      0.172     0.000     0.142     0.202
 (2 - 1)     -0.004     0.981    -0.033     0.025
 (2 - 3)      0.097     0.000     0.068     0.127
 (3 - 0)      0.075     0.000     0.045     0.104
 (3 - 1)     -0.102     0.000    -0.131    -0.072
 (3 - 2)     -0.097     0.000    -0.127    -0.068



In [18]:
print("N-2")
print(tukey_hsd(full_scores[2]["Human Train"], full_scores[2]["Gen V1"],full_scores[2]["Gen V2"], full_scores[2]["Gen V3"]))

N-2
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.236     0.000    -0.273    -0.200
 (0 - 2)     -0.223     0.000    -0.259    -0.186
 (0 - 3)     -0.094     0.000    -0.131    -0.057
 (1 - 0)      0.236     0.000     0.200     0.273
 (1 - 2)      0.014     0.765    -0.022     0.050
 (1 - 3)      0.142     0.000     0.106     0.178
 (2 - 0)      0.223     0.000     0.186     0.259
 (2 - 1)     -0.014     0.765    -0.050     0.022
 (2 - 3)      0.129     0.000     0.093     0.165
 (3 - 0)      0.094     0.000     0.057     0.131
 (3 - 1)     -0.142     0.000    -0.178    -0.106
 (3 - 2)     -0.129     0.000    -0.165    -0.093



In [19]:
print("N-3")
print(tukey_hsd(full_scores[3]["Human Train"], full_scores[3]["Gen V1"],full_scores[3]["Gen V2"], full_scores[3]["Gen V3"]))

N-3
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.273     0.000    -0.317    -0.228
 (0 - 2)     -0.250     0.000    -0.294    -0.205
 (0 - 3)     -0.109     0.000    -0.153    -0.065
 (1 - 0)      0.273     0.000     0.228     0.317
 (1 - 2)      0.023     0.525    -0.021     0.066
 (1 - 3)      0.164     0.000     0.120     0.207
 (2 - 0)      0.250     0.000     0.205     0.294
 (2 - 1)     -0.023     0.525    -0.066     0.021
 (2 - 3)      0.141     0.000     0.097     0.184
 (3 - 0)      0.109     0.000     0.065     0.153
 (3 - 1)     -0.164     0.000    -0.207    -0.120
 (3 - 2)     -0.141     0.000    -0.184    -0.097



In [20]:
print("N-4")
print(tukey_hsd(full_scores[4]["Human Train"], full_scores[4]["Gen V1"],full_scores[4]["Gen V2"], full_scores[4]["Gen V3"]))

N-4
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.294     0.000    -0.345    -0.244
 (0 - 2)     -0.264     0.000    -0.314    -0.213
 (0 - 3)     -0.120     0.000    -0.171    -0.069
 (1 - 0)      0.294     0.000     0.244     0.345
 (1 - 2)      0.030     0.387    -0.019     0.080
 (1 - 3)      0.174     0.000     0.125     0.224
 (2 - 0)      0.264     0.000     0.213     0.314
 (2 - 1)     -0.030     0.387    -0.080     0.019
 (2 - 3)      0.144     0.000     0.094     0.193
 (3 - 0)      0.120     0.000     0.069     0.171
 (3 - 1)     -0.174     0.000    -0.224    -0.125
 (3 - 2)     -0.144     0.000    -0.193    -0.094



In [21]:
print("N-5")
print(tukey_hsd(full_scores[5]["Human Train"], full_scores[5]["Gen V1"],full_scores[5]["Gen V2"], full_scores[5]["Gen V3"]))

N-5
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.309     0.000    -0.365    -0.254
 (0 - 2)     -0.272     0.000    -0.328    -0.217
 (0 - 3)     -0.130     0.000    -0.185    -0.074
 (1 - 0)      0.309     0.000     0.254     0.365
 (1 - 2)      0.037     0.298    -0.017     0.091
 (1 - 3)      0.180     0.000     0.125     0.234
 (2 - 0)      0.272     0.000     0.217     0.328
 (2 - 1)     -0.037     0.298    -0.091     0.017
 (2 - 3)      0.143     0.000     0.088     0.197
 (3 - 0)      0.130     0.000     0.074     0.185
 (3 - 1)     -0.180     0.000    -0.234    -0.125
 (3 - 2)     -0.143     0.000    -0.197    -0.088



# Self-BLEURT

In [73]:
# Install BLEURT: pip install git+https://github.com/google-research/bleurt.git
# GET BLEURT MODEL: wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip .

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

: 

In [ ]:
from bleurt import score

: 

In [ ]:
checkpoint = "BLEURT-20"
scorer = score.BleurtScorer(checkpoint)

: 

In [ ]:
import itertools

def calculate_self_bleurt(node: DialogNode) -> float:
    questions = set([q.text for q in node.questions])
    scores = []
    # do pair-wise 
    # for reference, hypothesis in itertools.combinations(questions, 2):
    #     score = scorer.score(references=[reference], candidates=[hypothesis])
    #     scores.extend(score)
 
    # do all at once
    # print(mean(scores))
    references, hypotheses = zip(*itertools.combinations(questions, 2))
    scores = scorer.score(references=references, candidates=hypotheses)
    
    return mean(scores)


: 

In [42]:
print("\n\n\n")
print("BLEURT")
datasets = {
    "Human Train": human_data_train,
    # "Human Test": human_data_test,
    "Gen V1": generated_data_train_v1,
    "Gen V2": generated_data_train_v2,
    "Gen V3": generated_data_train_v3
}

NGRAMS = [1,2,3,4,5]


ngram_scores = {}
for ngram in tqdm(NGRAMS):
    dataset_scores = {}
    for dataset_name in datasets:
        scores = []
        for node in datasets[dataset_name].nodes_by_type[NodeType.INFO]:
            if len(node.questions) <= 1:
                continue
            scores.append(calculate_self_bleurt(node))
        dataset_scores[dataset_name] = mean(scores)
    ngram_scores[ngram] = dataset_scores
print(ngram_scores)

  0%|          | 0/5 [1:03:04<?, ?it/s]


KeyboardInterrupt: 